# SENTINEL — Notebook 01: Dataset Parsing
Drain parsing + session construction for HDFS, BGL, OpenStack.

**Run Notebook 00 first.**

In [ ]:
import sys
sys.path.insert(0, '/content/SENTINEL')

from sentinel.data.parser import (
    DrainParser,
    build_sessions_hdfs,
    build_sessions_bgl,
    build_sessions_openstack,
    save_parsed, load_parsed,
)
from configs.config import cfg
import os

print('Imports OK ✓')

## 1.1 — HDFS Dataset

In [ ]:
# ── Paths ──
HDFS_LOG   = f'{cfg.DATA_DIR}/HDFS/HDFS.log'
HDFS_LABEL = f'{cfg.DATA_DIR}/HDFS/anomaly_label.csv'
HDFS_OUT   = f'{cfg.PARSED_DIR}/hdfs_parsed.pkl'
HDFS_VOCAB = f'{cfg.PARSED_DIR}/hdfs_parser.pkl'

# Check files exist
for p in [HDFS_LOG, HDFS_LABEL]:
    print(f'{"✓" if os.path.exists(p) else "✗ MISSING"} {p}')

In [ ]:
# ── Parse (skip if already done) ──
if not os.path.exists(HDFS_OUT):
    parser_hdfs = DrainParser(**cfg.DRAIN_SETTINGS['HDFS'])
    sessions, labels, block_ids = build_sessions_hdfs(
        HDFS_LOG, HDFS_LABEL, parser_hdfs,
        max_seq_len=cfg.MAX_SEQ_LEN,
        min_seq_len=cfg.MIN_SEQ_LEN,
    )
    save_parsed(sessions, labels, HDFS_OUT, extra={'block_ids': block_ids})
    parser_hdfs.save(HDFS_VOCAB)
    print(f'Vocab size: {parser_hdfs.vocab_size}')
else:
    print('HDFS already parsed — loading...')
    obj = load_parsed(HDFS_OUT)
    sessions, labels = obj['sessions'], obj['labels']

print(f'Total sessions: {len(sessions)}')
print(f'Sample session (first 10 events): {sessions[0][:10]}')
print(f'Sample label: {labels[0]}')

## 1.2 — BGL Dataset

In [ ]:
BGL_LOG  = f'{cfg.DATA_DIR}/BGL/BGL.log'
BGL_OUT  = f'{cfg.PARSED_DIR}/bgl_parsed.pkl'
BGL_VOCAB = f'{cfg.PARSED_DIR}/bgl_parser.pkl'

if not os.path.exists(BGL_OUT):
    parser_bgl = DrainParser(**cfg.DRAIN_SETTINGS['BGL'])
    sessions_bgl, labels_bgl = build_sessions_bgl(
        BGL_LOG, parser_bgl,
        window_size=cfg.SESSION_SETTINGS['BGL']['window'],
        stride=cfg.SESSION_SETTINGS['BGL']['stride'],
        max_seq_len=cfg.MAX_SEQ_LEN,
    )
    save_parsed(sessions_bgl, labels_bgl, BGL_OUT)
    parser_bgl.save(BGL_VOCAB)
else:
    print('BGL already parsed — loading...')
    obj = load_parsed(BGL_OUT)
    sessions_bgl, labels_bgl = obj['sessions'], obj['labels']

print(f'Total BGL sessions: {len(sessions_bgl)}')

## 1.3 — OpenStack Dataset

In [ ]:
OS_LOG_DIR = f'{cfg.DATA_DIR}/OpenStack/'
OS_OUT     = f'{cfg.PARSED_DIR}/openstack_parsed.pkl'
OS_VOCAB   = f'{cfg.PARSED_DIR}/openstack_parser.pkl'

if not os.path.exists(OS_OUT):
    parser_os = DrainParser(**cfg.DRAIN_SETTINGS['OpenStack'])
    sessions_os, labels_os, req_ids = build_sessions_openstack(
        OS_LOG_DIR, parser_os,
        max_seq_len=cfg.MAX_SEQ_LEN,
    )
    save_parsed(sessions_os, labels_os, OS_OUT, extra={'req_ids': req_ids})
    parser_os.save(OS_VOCAB)
else:
    print('OpenStack already parsed — loading...')
    obj = load_parsed(OS_OUT)
    sessions_os, labels_os = obj['sessions'], obj['labels']

print(f'Total OpenStack sessions: {len(sessions_os)}')

## 1.4 — Summary

In [ ]:
import pandas as pd

def dataset_summary(name, sessions, labels):
    n_total   = len(sessions)
    n_normal  = sum(1 for l in labels if l == 0)
    n_anomaly = sum(1 for l in labels if l == 1)
    avg_len   = sum(len(s) for s in sessions) / n_total
    return {
        'Dataset':   name,
        'Sessions':  n_total,
        'Normal':    n_normal,
        'Anomalous': n_anomaly,
        'Anomaly %': f'{100*n_anomaly/n_total:.1f}%',
        'Avg Len':   f'{avg_len:.1f}',
    }

summary = pd.DataFrame([
    dataset_summary('HDFS',      sessions,    labels),
    dataset_summary('BGL',       sessions_bgl, labels_bgl),
    dataset_summary('OpenStack', sessions_os,  labels_os),
])
print(summary.to_string(index=False))
print('\nParsing complete ✓  Proceed to Notebook 02 → Training')